# Credit Card Fraud Analytics — SQL Analysis

## Phase 7: SQL Analysis

This phase translates the project's analytical questions into SQL.

Objectives:
- Recreate data-quality checks in SQL
- Calculate fraud KPIs
- Segment transactions by amount and hour
- Rank risk segments
- Produce reusable SQL outputs for a Data Analyst portfolio

The notebook uses SQLite so the SQL can be executed locally without a separate database server.


## 1. Load the Clean Dataset into SQLite


In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

DATA_PATH = "../data/creditcard_clean.csv"

df = pd.read_csv(DATA_PATH)

connection = sqlite3.connect(":memory:")
df.to_sql("credit_card_transactions", connection, index=False, if_exists="replace")

print(f"Rows loaded: {len(df):,}")
print("Table: credit_card_transactions")


## 2. Inspect the SQL Table


In [ ]:
table_info = pd.read_sql_query(
    "PRAGMA table_info(credit_card_transactions);",
    connection
)

display(table_info)


## 3. Data Overview


In [ ]:
query = '''
SELECT
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS fraud_transactions,
    ROUND(
        100.0 * SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS fraud_rate_pct
FROM credit_card_transactions;
'''

display(pd.read_sql_query(query, connection))


## 4. Fraud KPIs


In [ ]:
query = '''
SELECT
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS fraud_transactions,
    ROUND(
        100.0 * SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS fraud_rate_pct,
    ROUND(SUM(Amount), 2) AS total_transaction_amount,
    ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) AS fraud_amount,
    ROUND(
        100.0 * SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END)
        / NULLIF(SUM(Amount), 0),
        4
    ) AS fraud_amount_share_pct
FROM credit_card_transactions;
'''

fraud_kpis = pd.read_sql_query(query, connection)
display(fraud_kpis)


## 5. Fraud vs Normal Amount Profile


In [ ]:
query = '''
SELECT
    CASE WHEN Class = 1 THEN 'Fraud' ELSE 'Normal' END AS transaction_class,
    COUNT(*) AS transactions,
    ROUND(AVG(Amount), 2) AS average_amount,
    ROUND(MIN(Amount), 2) AS minimum_amount,
    ROUND(MAX(Amount), 2) AS maximum_amount,
    ROUND(SUM(Amount), 2) AS total_amount
FROM credit_card_transactions
GROUP BY Class
ORDER BY Class;
'''

amount_profile = pd.read_sql_query(query, connection)
display(amount_profile)


## 6. Fraud Rate by Amount Band


In [ ]:
query = '''
WITH banded AS (
    SELECT
        Class,
        Amount,
        CASE
            WHEN Amount <= 10 THEN '0-10'
            WHEN Amount <= 25 THEN '10-25'
            WHEN Amount <= 50 THEN '25-50'
            WHEN Amount <= 100 THEN '50-100'
            WHEN Amount <= 250 THEN '100-250'
            WHEN Amount <= 500 THEN '250-500'
            WHEN Amount <= 1000 THEN '500-1000'
            WHEN Amount <= 2500 THEN '1000-2500'
            WHEN Amount <= 5000 THEN '2500-5000'
            ELSE '5000+'
        END AS amount_band
    FROM credit_card_transactions
)
SELECT
    amount_band,
    COUNT(*) AS transactions,
    SUM(Class) AS fraud_transactions,
    ROUND(100.0 * SUM(Class) / COUNT(*), 4) AS fraud_rate_pct
FROM banded
GROUP BY amount_band
ORDER BY
    CASE amount_band
        WHEN '0-10' THEN 1
        WHEN '10-25' THEN 2
        WHEN '25-50' THEN 3
        WHEN '50-100' THEN 4
        WHEN '100-250' THEN 5
        WHEN '250-500' THEN 6
        WHEN '500-1000' THEN 7
        WHEN '1000-2500' THEN 8
        WHEN '2500-5000' THEN 9
        ELSE 10
    END;
'''

amount_segments = pd.read_sql_query(query, connection)
display(amount_segments)


## 7. Fraud Rate by Hour of Day


In [ ]:
query = '''
WITH hourly AS (
    SELECT
        CAST((Time / 3600) AS INTEGER) % 24 AS hour_of_day,
        Class,
        Amount
    FROM credit_card_transactions
)
SELECT
    hour_of_day,
    COUNT(*) AS transactions,
    SUM(Class) AS fraud_transactions,
    ROUND(100.0 * SUM(Class) / COUNT(*), 4) AS fraud_rate_pct,
    ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) AS fraud_amount
FROM hourly
GROUP BY hour_of_day
ORDER BY hour_of_day;
'''

hourly_segments = pd.read_sql_query(query, connection)
display(hourly_segments)


## 8. Rank Hours by Fraud Rate


In [ ]:
query = '''
WITH hourly AS (
    SELECT
        CAST((Time / 3600) AS INTEGER) % 24 AS hour_of_day,
        COUNT(*) AS transactions,
        SUM(Class) AS fraud_transactions
    FROM credit_card_transactions
    GROUP BY CAST((Time / 3600) AS INTEGER) % 24
)
SELECT
    hour_of_day,
    transactions,
    fraud_transactions,
    ROUND(100.0 * fraud_transactions / transactions, 4) AS fraud_rate_pct
FROM hourly
ORDER BY fraud_rate_pct DESC, transactions DESC;
'''

hourly_ranking = pd.read_sql_query(query, connection)
display(hourly_ranking)


## 9. High-Value Transaction Segment


In [ ]:
query = '''
SELECT
    CASE WHEN Amount >= 1000 THEN '1000+' ELSE '<1000' END AS value_segment,
    COUNT(*) AS transactions,
    SUM(Class) AS fraud_transactions,
    ROUND(100.0 * SUM(Class) / COUNT(*), 4) AS fraud_rate_pct,
    ROUND(SUM(Amount), 2) AS total_amount,
    ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) AS fraud_amount
FROM credit_card_transactions
GROUP BY value_segment
ORDER BY fraud_rate_pct DESC;
'''

high_value = pd.read_sql_query(query, connection)
display(high_value)


## 10. Zero-Amount Segment


In [ ]:
query = '''
SELECT
    CASE WHEN Amount = 0 THEN 'Zero Amount' ELSE 'Non-Zero Amount' END AS amount_status,
    COUNT(*) AS transactions,
    SUM(Class) AS fraud_transactions,
    ROUND(100.0 * SUM(Class) / COUNT(*), 4) AS fraud_rate_pct
FROM credit_card_transactions
GROUP BY amount_status
ORDER BY fraud_rate_pct DESC;
'''

zero_amount = pd.read_sql_query(query, connection)
display(zero_amount)


## 11. SQL-Based Business Questions

The SQL layer answers:

1. What is the overall fraud rate?
2. What share of transaction amount is associated with fraud?
3. Which amount bands have higher observed fraud rates?
4. Which hours have higher observed fraud rates?
5. How does zero-amount behavior compare with non-zero transactions?
6. How does the high-value transaction segment compare with the rest of the population?

These queries describe observed associations and do not establish causality.


## 12. Save SQL Outputs


In [ ]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

fraud_kpis.to_csv(OUTPUT_DIR / "sql_fraud_kpis.csv", index=False)
amount_profile.to_csv(OUTPUT_DIR / "sql_amount_profile.csv", index=False)
amount_segments.to_csv(OUTPUT_DIR / "sql_amount_segments.csv", index=False)
hourly_segments.to_csv(OUTPUT_DIR / "sql_hourly_segments.csv", index=False)
hourly_ranking.to_csv(OUTPUT_DIR / "sql_hourly_ranking.csv", index=False)
high_value.to_csv(OUTPUT_DIR / "sql_high_value_segment.csv", index=False)
zero_amount.to_csv(OUTPUT_DIR / "sql_zero_amount_segment.csv", index=False)

print("SQL analysis outputs saved successfully.")


## 13. Phase 7 Conclusions

The SQL layer provides a reproducible business-analysis workflow:

- Data quality can be checked with SQL.
- Core fraud KPIs can be calculated with SQL.
- Fraud can be segmented by transaction amount and hour.
- Risk segments can be ranked using SQL.
- The resulting tables can be consumed by Power BI.

### Next Phase

**Phase 8 — Power BI Dashboard**

The next phase will transform the analytical outputs into an executive-style fraud analytics dashboard.
